# Hill Climbing

Hill climbing is an ensemble technique to combine multiple models. We start with the best-performing model, then add models one by one with different weights, keeping any addition that improves the blended CV score. We repeat until no addition helps.

Our metric is AUC (higher is better), but `hill_climb_ensemble` *minimizes* its metric. So we pass `1 - AUC` — minimizing AUC-error is the same as maximizing AUC.

In [ ]:
VER = 1

## Load Data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

train = pd.read_parquet('data/train_features.parquet')
y_train_true = train['PitNextLap'].astype(int).values
print('Train shape:', train.shape)

## Model Performance EDA

Our current models and their OOF AUC CV scores — 4 baselines from `03` and 2 stacking models from `04`.

In [ ]:
model_names = ['linear', 'xgb', 'lgb', 'cb', 'stack_target', 'stack_feature']
oof_list, test_list = [], []
for k in model_names:
    oof_list.append(np.load(f'data/train_oof_{k}_v{VER}.npy'))
    test_list.append(np.load(f'data/test_pred_{k}_v{VER}.npy').mean(0))

aucs = [roc_auc_score(y_train_true, oof) for oof in oof_list]
for m, a in zip(model_names, aucs):
    print(f'{m:14s} model has {a:.5f} OOF AUC')

In [ ]:
# Plot AUC-error (1 - AUC) on a log scale: lower is better, like the RMSE chart in the template.
errs = [1 - a for a in aucs]
plt.figure(figsize=(8, 4))
plt.bar(model_names, errs)
plt.ylabel('log10 (1 - AUC)  [lower is better]')
plt.xlabel('Model')
plt.title('Model AUC-Error Comparison')
plt.xticks(rotation=45, ha='right')
plt.yscale('log')
plt.tight_layout()
plt.show()

## Hill Climbing

We apply hill climbing to the 6 models. It begins with the most accurate model and then tries blending additional models one by one to improve CV AUC.

In [ ]:
from data.utils import hill_climb_ensemble

def neg_auc(y_true, y_pred):
    # hill_climb_ensemble minimizes its metric; minimizing (1 - AUC) maximizes AUC.
    return 1.0 - roc_auc_score(y_true, y_pred)

result = hill_climb_ensemble(
    oofs=oof_list,
    test_preds=test_list,
    names=model_names,
    y_true=y_train_true,
    metric=neg_auc,
    max_number_models=None,
    tolerance=1e-6,
    use_negative_weights=False,
)

In [ ]:
oof_auc = roc_auc_score(y_train_true, result['oof_pred'])
print(f'Hill-Climb Ensemble, CV OOF AUC: {oof_auc:.5f}')
print('weights:', dict(zip(result['used_names'], np.round(result['weights'], 4))))

## Ensemble Performance EDA

The hill-climbing ensemble should sit below every individual model on AUC-error.

In [ ]:
model_names2 = model_names + ['hill_climb']
errs2 = [1 - a for a in aucs] + [1 - oof_auc]
plt.figure(figsize=(8, 4))
plt.bar(model_names2, errs2)
plt.ylabel('log10 (1 - AUC)  [lower is better]')
plt.xlabel('Model')
plt.title('Model AUC-Error Comparison (with hill-climb)')
plt.xticks(rotation=45, ha='right')
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Save the hill-climb ensemble OOF/PRED for the downstream notebooks.
np.save(f'data/train_oof_hill_climb_v{VER}.npy', result['oof_pred'])
np.save(f'data/test_pred_hill_climb_v{VER}.npy', result['test_pred'][None, :])